In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.append("..")
from src.data.preprocessing import build_leave_one_out
from src.data.dataset import SessionDataset, collate_sessions
from src.models.sasrec import SASRec
from src.losses.sampled_ranking_loss import BPRLoss
from src.trainers.sasrec_trainer import SASRecTrainer

In [3]:
PROCESSED_DIR = Path("../data/processed")
DEVICE = torch.device("cpu")

with open(PROCESSED_DIR / "sequences.pkl", "rb") as f:
    sequences = pickle.load(f)

with open(PROCESSED_DIR / "item_mappings.pkl", "rb") as f:
    mappings = pickle.load(f)
vocab_size = mappings["vocab_size"]

split_data = build_leave_one_out(sequences)
train_prefixes = {uid: data["train"] for uid, data in split_data.items()}

train_lengths = [len(seq) for seq in train_prefixes.values()]
MAX_LEN = int(np.percentile(train_lengths, 99))
print(f"Global max_len (99th percentile): {MAX_LEN}")

Global max_len (99th percentile): 1026


In [4]:
split_data = build_leave_one_out(sequences)
print(f"After split: {len(split_data)} users")

train_prefixes = {uid: data["train"] for uid, data in split_data.items()}

After split: 197519 users


In [5]:
dataset = SessionDataset(train_prefixes, max_len=MAX_LEN)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_sessions)
print(f"Number of batches: {len(dataloader)}")

Number of batches: 6173


In [6]:
embed_dim = 64
num_layers = 2
nhead = 4
num_negs = 10
lr = 1e-3
weight_decay = 0.001
epochs = 10

model = SASRec(vocab_size, embed_dim, MAX_LEN, num_layers, nhead).to(DEVICE)
bpr_loss = BPRLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

trainer = SASRecTrainer(model, bpr_loss, optimizer, vocab_size, DEVICE, num_negs=num_negs)

In [7]:
sum(p.numel() for p in model.parameters())

3236096

In [ ]:
best_recall = -1
best_model_state = None

for epoch in range(1, epochs + 1):
    loss = trainer.train_epoch(dataloader)
    val_metrics = trainer.evaluate(split_data, target_key="val_target", k=5)
    recall = val_metrics["Recall@5"]
    print(f"Epoch {epoch}: loss=j{loss:.4f}, val Recall@5={recall:.4f}, val NDCG@5={val_metrics['NDCG@5']:.4f}, val MRR={val_metrics['MRR']:.4f}")

    if recall > best_recall:
        best_recall = recall
        best_model_state = model.state_dict().copy()
        torch.save(best_model_state, PROCESSED_DIR / "sasrec_best.pt")
        print(f"New best model saved (Recall@5={best_recall:.4f})")